In [1]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

train_df = pd.read_csv("preprocessed_train_final.csv")
test_df  = pd.read_csv("preprocessed_test_final.csv")
train_df_2 = pd.read_csv("preprocessed_train_LSH.csv")


c:\Users\ilker\anaconda3\envs\torchgpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ilker\AppData\Local\Temp\ipykernel_25192\3455001347.py:6: DtypeWarning: Columns (0: product_id) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df  = pd.read_csv("preprocessed_test_final.csv")


In [3]:
train_df.head(5)

,product_id,product_text,category,product_text_clean,dominant_category,purity,total_cnt,canon_text,text_for_model,char_len,...,is_coarse,desc_token,template_key_ref,grank,cat_cnt,cap_cat,rank_cat,gsize,cap_key,rank_key
0,106588455,Deniz Feneri Iskele Kanvas Tablo 180x60cm,Tablo,deniz feneri iskele kanvas tablo 180x60cm,Tablo,1.0,1.0,deniz feneri iskele kanvas tablo 180x60cm,deniz feneri iskele kanvas tablo 180x60cm,41,...,False,NaN,deniz feneri iskele,2,25451,8000,0,4,999999999,0
1,85876069,Opel Corsa Serisi Silver-gold Model Ön Arka Ot...,Oto Koltuk Kılıfı,opel corsa serisi silver gold model ön arka ot...,Oto Koltuk Kılıfı,1.0,1.0,opel corsa serisi silver gold model on arka ot...,opel corsa serisi silver gold model ön arka ot...,61,...,False,NaN,opel corsa,9,18394,12000,0,110,999999999,0
2,84858509,Ilk Kitaplarım Şekiller,Bebek & Aktivite Oyuncakları,ilk kitaplarım şekiller,Bebek & Aktivite Oyuncakları,1.0,1.0,ilk kitaplarim sekiller,ilk kitaplarım şekiller,23,...,False,NaN,ilk kitaplarim,1,2014,999999999,0,3,999999999,0
3,54853260,Yeşil Nokta Desen Mendilli Klasik Kravat Kk10027,Kravat,yeşil nokta desen mendilli klasik kravat kk10027,Kravat,1.0,1.0,yesil nokta desen mendilli klasik kravat kk10027,yeşil nokta desen mendilli klasik kravat kk10027,48,...,False,NaN,nokta desen,78,5259,12000,0,123,999999999,0
4,198720781,Mor Zeminde Pembe Yuvarlak Desenli Dekoratif (...,Fon Perde,mor zeminde pembe yuvarlak desenli dekoratif t...,Fon Perde,1.0,2.0,mor zeminde pembe yuvarlak desenli dekoratif t...,mor zeminde pembe yuvarlak desenli dekoratif t...,72,...,False,NaN,zeminde yuvarlak dekoratif,20,19263,12000,0,16,999999999,0


In [4]:
train_df.columns

Index(['product_id', 'product_text', 'category', 'product_text_clean',
       'dominant_category', 'purity', 'total_cnt', 'canon_text',
       'text_for_model', 'char_len', 'tok_cnt', 'is_junk', 'cluster_id',
       'sample_weight', 'main_category', 'hash_text', 'fold2', 'mismatch_oof',
       'pred_category_oof', 'gap_oof', 'margin_oof', 'flag_suspect',
       'category_fixed', 'main_category_old', 'template_key', 'cnt',
       'is_coarse', 'desc_token', 'template_key_ref', 'grank', 'cat_cnt',
       'cap_cat', 'rank_cat', 'gsize', 'cap_key', 'rank_key'],
      dtype='str')

In [5]:


# Choose text column
TEXT_COL = "text_for_model"

LABEL_TRAIN = "category_fixed"

train_df[TEXT_COL] = train_df[TEXT_COL].fillna("").astype(str)
test_df[TEXT_COL]  = test_df[TEXT_COL].fillna("").astype(str)

print("Train:", train_df.shape, "| Test:", test_df.shape)
print("Train label col:", LABEL_TRAIN, "| Text col:", TEXT_COL)


Train: (2872653, 36) | Test: (1082903, 12)
Train label col: category_fixed | Text col: text_for_model


In [14]:
test_df = test_df.dropna(subset="category").reset_index(drop=True)


In [16]:
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack

# Light clean (punct->space) for hashing model
def to_hash_text(s: pd.Series) -> pd.Series:
    return (s.str.lower()
              .str.replace(r"[^\w\sğüşöçıİĞÜŞÖÇ]", " ", regex=True)
              .str.replace(r"\s+", " ", regex=True)
              .str.strip())

train_used["hash_text"] = to_hash_text(train_used[TEXT_COL])
test_df["hash_text"]    = to_hash_text(test_df[TEXT_COL])

word_vec = HashingVectorizer(
    n_features=2**19, alternate_sign=False, norm="l2",
    analyzer="word", ngram_range=(1,2), lowercase=False
)
char_vec = HashingVectorizer(
    n_features=2**18, alternate_sign=False, norm="l2",
    analyzer="char_wb", ngram_range=(3,5), lowercase=False
)

def X_transform(text_series):
    return hstack([word_vec.transform(text_series), char_vec.transform(text_series)], format="csr")

# Label encoding
le = LabelEncoder()
y_train = le.fit_transform(train_used[LABEL_TRAIN].astype(str).values)
y_test  = le.transform(test_df["category"].astype(str).values)  # assumes no new categories

n_classes = len(le.classes_)
print("Classes:", n_classes)


Classes: 1140


In [18]:
from sklearn.linear_model import SGDClassifier
from scipy.special import logsumexp
import numpy as np
from tqdm.auto import tqdm

def batch_logloss_from_scores(S, y, sample_weight=None):
    """
    Multi-class log-loss computed from raw scores S (decision_function output).
    loss_i = - (s_true - logsumexp(scores))
    """
    if S.ndim == 1:  # binary guard
        S = np.vstack([-S, S]).T

    lse = logsumexp(S, axis=1)
    s_true = S[np.arange(len(y)), y]
    loss = -(s_true - lse)

    if sample_weight is None:
        return float(loss.mean())
    sw = sample_weight.astype(np.float64)
    return float(np.sum(sw * loss) / np.sum(sw))

clf = SGDClassifier(
    loss="log_loss",
    alpha=5e-6,
    penalty="l2",
    max_iter=1,
    tol=None,
    n_jobs=-1,
    random_state=42,
    average=True
)

BATCH = 50_000
EPOCHS = 2
PRINT_EVERY = 20   # print every N batches (e.g., 10, 20, 50)
rng = np.random.RandomState(42)

idx = np.arange(len(train_used))
classes_int = np.arange(n_classes, dtype=np.int32)

global_step = 0
running_loss = 0.0
running_w = 0.0

for ep in range(EPOCHS):
    rng.shuffle(idx)
    for start in tqdm(range(0, len(idx), BATCH), desc=f"Train epoch {ep+1}/{EPOCHS}"):
        b = idx[start:start+BATCH]

        Xb = X_transform(train_used["hash_text"].iloc[b])
        yb = y_train[b]
        wb = train_used["sample_weight"].iloc[b].values.astype(np.float32)

        # update
        if ep == 0 and start == 0:
            clf.partial_fit(Xb, yb, classes=classes_int, sample_weight=wb)
        else:
            clf.partial_fit(Xb, yb, sample_weight=wb)

        global_step += 1

        # print loss sometimes (on current batch, after update)
        if global_step % PRINT_EVERY == 0:
            S = clf.decision_function(Xb)
            loss_val = batch_logloss_from_scores(S, yb, wb)

            # optional: batch accuracy for sanity
            if np.ndim(S) == 1:
                S = np.vstack([-S, S]).T
            pred_b = S.argmax(axis=1)
            acc_b = float((pred_b == yb).mean())

            # running weighted loss (rough)
            running_loss = 0.9 * running_loss + 0.1 * loss_val
            print(f"[ep {ep+1}] step {global_step:>6} | batch_logloss={loss_val:.4f} | batch_acc={acc_b:.4f} | smooth_loss={running_loss:.4f}")

print("Training done.")


Train epoch 1/2:   2%|▏         | 1/58 [07:08<6:47:10, 428.61s/it]


KeyboardInterrupt: 